# Sampling-rate sensitivity — single-trajectory deep dive

Visual companion to `sampling_sensitivity.py`. The native simulator grid is **2000 Hz**; training the PINN there is costly (seq_len, memory, throughput), so we want to know whether **1000 Hz** or **500 Hz** would lose anything that matters. This notebook takes **one trajectory** and shows, step by step, what each downsampling choice does to it.

## The two ways to downsample — opposite failure modes

| mode | what it does | failure mode |
|---|---|---|
| **anti-aliased decimation** | low-pass, then keep every k-th sample | content above the new Nyquist is **lost** (cleanly) |
| **naive subsampling** (`x[::k]`) | just keep every k-th sample | content above the new Nyquist **folds back** into the band — fake low-frequency structure |

Because ASMC chatter and numerical hash live at high frequency, **naive subsampling can manufacture contamination** the chatter screen then (correctly) flags. So this study is also a check on *how* the PINN data loader should downsample, not just *how far*.

## How we decide 'significant'
1. **Force reconstruction error** — band-limited (`resample_poly`) reconstruction of the resampled force back to 2000 Hz vs native, per family, normalised. Band-limited (not linear) reconstruction isolates genuine information loss/aliasing from interpolator crudeness. Compare against the interpolation floor (~2.7e-4 for Mz).
2. **Chatter-verdict flips** — does the chatter screen's verdict change across rate/mode?

## Physics that bounds the answer
- Roller ripple lives at `h·f_roll = h·1.910·|ω|` Hz — bounded by wheel speed (3rd harmonic ≤ 86 Hz at |ω|=15). Essentially nothing above 500 Hz.
- LuGre stick–slip is a slip-driven low-pass with corner `f_lugre ≈ 653·Vp` Hz — this is the content that can reach high frequency at high slip.

So the expectation: **1000 Hz (Nyquist 500) is low-risk; 500 Hz (Nyquist 250) only bites high-slip / high-ω regimes.** This notebook lets you confirm it per trajectory.

In [ ]:
import importlib
import tempfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pyarrow.feather as feather
from scipy.signal import welch, resample_poly

import chatter_diagnostics as cd
import sampling_sensitivity as ss
importlib.reload(cd); importlib.reload(ss)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.titlesize': 11, 'legend.fontsize': 8})
cfg = cd.ChatterConfig()

## 1. Pick a trajectory

Same convention as the chatter deep-dive. Point `DATA_DIR`/`TARGET_FILE` at real sweep output, or leave it to fall back to a synthetic trajectory of type `DEMO_KIND`. For *this* study the most instructive synthetic is `lugre_above` — it deliberately carries content above 250 Hz, so you can watch 500 Hz break while 1000 Hz holds. `clean` (all energy < 50 Hz) should survive every rate untouched.

In [ ]:
# === EDIT THESE ===
DATA_DIR    = Path(r'..\data\Simulation_Data_MecanumSlipSpin_LugreAdamov')
TARGET_FILE = None              # None -> first arrow in DATA_DIR; else a filename
WHEEL       = 0                 # 0..3 — which wheel to visualise
RATE_FOCUS  = 500               # the rate used for the time/PSD overlays (500 or 1000)
DEMO_KIND   = 'lugre_above'     # synthetic fallback: clean|chatter|hash|lugre_heavy|lugre_above
# ===================

_NAME_BY_KIND = {'clean': 'octagon', 'chatter': 'spin_creep', 'hash': 'coupled_vomega',
                 'lugre_heavy': 'spiral_orbit', 'lugre_above': 'long_circle'}

paths = sorted(DATA_DIR.glob('*.arrow')) if DATA_DIR.exists() else []
paths = [p for p in paths if cd.parse_arrow_filename(p.name)]
if TARGET_FILE:
    path = DATA_DIR / TARGET_FILE
elif paths:
    path = paths[0]
else:
    import test_chatter_diagnostics as tt
    _tmp = Path(tempfile.mkdtemp())
    path = _tmp / (_NAME_BY_KIND[DEMO_KIND] + '_c001_mu_0.4_case1_lugre_adamov_chi_0.005.arrow')
    feather.write_feather(tt.make_traj(DEMO_KIND), path)
    print('[demo] no real data found -> synthetic', repr(DEMO_KIND))

t, W, omega, theta = cd.load_columns(path)
meta = cd.parse_arrow_filename(path.name)
fs = 1.0 / float(np.median(np.diff(t)))
i, mu, chi = WHEEL, meta['mu'], meta['chi']
print('file:', path.name)
print(f'fs = {fs:.0f} Hz   N = {len(t)}   wheel = {i + 1}   focus rate = {RATE_FOCUS} Hz')

## 2. Where the force energy lives vs the candidate Nyquists

The whole question reduces to: *how much force energy sits above each candidate Nyquist?* The Nyquist of rate r is r/2, so the three rates give cutoffs at **1000 Hz** (2000 Hz native), **500 Hz** (1000 Hz rate), **250 Hz** (500 Hz rate). Energy below a cutoff is safe at that rate; energy above it is what you either lose (anti-aliased) or alias (naive).

The plot is the native PSD of `Fpar` with the three Nyquists marked; the annotations give the fraction of off-DC energy above each (median over wheels, the same number `sampling_sensitivity.py` records). A faithful low rate is one where that fraction is negligible.

In [ ]:
f, P = welch(W['Fpar'][:, i], fs=fs, nperseg=min(cfg.welch_nperseg, len(t)), detrend='constant')
fig, ax = plt.subplots(figsize=(11, 4))
ax.semilogy(f, P + 1e-20, lw=0.7, label='native Fpar PSD')
for nyq, c in [(250, 'C3'), (500, 'C2'), (1000, 'C0')]:
    ax.axvline(nyq, color=c, ls='--', lw=1.2, label=f'Nyquist {nyq} Hz (rate {2 * nyq})')
for fam in ss.FAMILIES:
    fr = {nyq: ss._energy_above(W[fam], fs, nyq, cfg) for nyq in (250, 500, 1000)}
    print(f'  energy above Nyquist  {fam:<5}:  >250Hz {fr[250]*100:6.2f}%   '
          f'>500Hz {fr[500]*100:6.2f}%   >1000Hz {fr[1000]*100:6.2f}%')
ax.set_xlabel('freq (Hz)'); ax.set_ylabel('PSD of Fpar'); ax.legend(fontsize=8)
ax.set_title('Native force spectrum vs candidate Nyquists (energy fractions printed below)')
plt.show()

## 3. Anti-aliased vs naive in the time domain

A short zoomed window of `Fpar` at the focus rate, three traces:
- **native** (2000 Hz) — ground truth.
- **anti-aliased** — low-passed then decimated, reconstructed back to the native grid. It tracks the slow shape and the in-band ripple but **omits** anything above the new Nyquist (smoother).
- **naive** — index-sliced then reconstructed. Where the native signal has content above the new Nyquist, naive subsampling can't represent it and instead draws a **wrong, lower-frequency wiggle** — the visual signature of aliasing.

If the three traces sit on top of each other, the focus rate is faithful for this trajectory.

In [ ]:
k = int(round(fs / RATE_FOCUS))
ta, Wa, _, _ = ss._resample(t, W, omega, theta, k, 'antialias')
tn, Wn, _, _ = ss._resample(t, W, omega, theta, k, 'naive')
rec_a = resample_poly(Wa['Fpar'][:, i], k, 1)
rec_n = resample_poly(Wn['Fpar'][:, i], k, 1)
L = min(len(t), len(rec_a), len(rec_n))
w0 = L // 3; w1 = w0 + int(0.08 * fs)        # ~80 ms window
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t[w0:w1], W['Fpar'][w0:w1, i], lw=1.2, label='native 2000 Hz')
ax.plot(t[w0:w1], rec_a[w0:w1], lw=1.0, label=f'anti-aliased {RATE_FOCUS} Hz (recon)')
ax.plot(t[w0:w1], rec_n[w0:w1], lw=1.0, ls='--', label=f'naive {RATE_FOCUS} Hz (recon)')
ax.set_xlabel('time (s)'); ax.set_ylabel('Fpar (N)'); ax.legend()
ax.set_title(f'~80 ms zoom — native vs {RATE_FOCUS} Hz (anti-aliased vs naive), wheel {i + 1}')
plt.show()

## 4. Anti-aliased vs naive in the spectrum — what aliasing actually does

The clearest view of aliasing is spectral. All three PSDs are plotted on the same axis (each computed at its own rate, so the downsampled ones only extend to their Nyquist):
- **anti-aliased** matches the native PSD up to the new Nyquist, then simply stops — the content above is gone, not relocated.
- **naive** also stops at the new Nyquist, but its in-band curve sits **above** the native one wherever high-frequency energy has *folded down* into the band. That excess is the aliased energy — phantom in-band power that was never physically there at those frequencies.

Anti-aliased loses; naive corrupts. For the PINN, corrupted in-band targets are worse than honestly-missing high-frequency detail.

In [ ]:
fr_hz = RATE_FOCUS
fa, Pa = welch(Wa['Fpar'][:, i], fs=fr_hz, nperseg=min(2048, len(Wa['Fpar'])), detrend='constant')
fn, Pn = welch(Wn['Fpar'][:, i], fs=fr_hz, nperseg=min(2048, len(Wn['Fpar'])), detrend='constant')
fig, ax = plt.subplots(figsize=(11, 4))
ax.semilogy(f, P + 1e-20, lw=0.8, color='C7', label='native (2000 Hz)')
ax.semilogy(fa, Pa + 1e-20, lw=1.0, color='C2', label=f'anti-aliased ({fr_hz} Hz)')
ax.semilogy(fn, Pn + 1e-20, lw=1.0, color='C3', ls='--', label=f'naive ({fr_hz} Hz)')
ax.axvline(fr_hz / 2, color='k', ls=':', lw=1.0, label=f'Nyquist {fr_hz // 2:.0f} Hz')
ax.set_xlim(0, min(1000, fs / 2)); ax.set_xlabel('freq (Hz)'); ax.set_ylabel('PSD of Fpar')
ax.legend(); ax.set_title(f'Spectra: native vs anti-aliased vs naive at {fr_hz} Hz '
                          f'(naive > native below Nyquist = aliased energy)')
plt.show()

## 5. Significance criterion 1 — force reconstruction error

For each force family (`Fpar`, `Fperp`, `Mz`), each rate, and each mode: reconstruct the resampled signal back to the native grid (band-limited, `resample_poly`) and report the normalised RMS error vs native, median over wheels. Read it as *the fraction of the signal that downsampling could not faithfully carry*:
- **anti-aliased error** ≈ the energy genuinely above the new Nyquist (irrecoverable loss).
- **naive error** ≥ anti-aliased — it adds the aliasing on top.

A rate is **insignificant** for a family if its error is at or below the interpolation floor the PINN already tolerates (~2.7e-4 for Mz); **significant** if the error jumps (typically at 500 Hz for high-slip trajectories), and *especially* if naive >> anti-aliased (a strong argument the loader must anti-alias filter).

In [ ]:
rates = (1000, 500)
recon = {}
for r in rates:
    kk = int(round(fs / r))
    for mode in ('antialias', 'naive'):
        _, Wr, _, _ = ss._resample(t, W, omega, theta, kk, mode)
        for fam in ss.FAMILIES:
            recon[(r, mode, fam)] = ss._recon_error(W[fam], Wr[fam], kk)

print(f"{'rate/mode':<20} {'Fpar':>9} {'Fperp':>9} {'Mz':>9}")
for r in rates:
    for mode in ('antialias', 'naive'):
        vals = [recon[(r, mode, fam)] for fam in ss.FAMILIES]
        print(f"  {r:>4} Hz {mode:<11} {vals[0]:>9.4f} {vals[1]:>9.4f} {vals[2]:>9.4f}")

fig, ax = plt.subplots(figsize=(9, 3.6))
labels = [f'{r}Hz\n{m}' for r in rates for m in ('antialias', 'naive')]
x = np.arange(len(labels)); wbar = 0.26
for j, fam in enumerate(ss.FAMILIES):
    ax.bar(x + (j - 1) * wbar, [recon[(r, m, fam)] for r in rates for m in ('antialias', 'naive')],
           wbar, label=fam)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('normalised recon error')
ax.legend(); ax.set_title('Force reconstruction error by rate / mode / family')
plt.show()

## 6. Significance criterion 2 — chatter-verdict stability

Run the chatter screen on the trajectory **as it would look at each rate/mode** and compare to the native verdict. A flip is a red flag:
- **naive → `hash`** where native was `clean`: aliasing has manufactured high-frequency contamination. Direct evidence the loader must not naive-subsample.
- **anti-aliased flips**: the rate has removed enough physical structure to change the screen's mind — usually a sign the rate is too low for this regime.
- **no flips**: the screen sees the same trajectory at that rate — reassuring (though note that at 500 Hz the screen is *blind* above 250 Hz, so absence of a flip is necessary, not sufficient).

In [ ]:
srow = ss.analyze_file(path, cfg, rates=(1000.0, 500.0))
print(f"native verdict: {srow['verdict_native']}\n")
print(f"{'rate/mode':<20} {'verdict':>10} {'flip?':>7}")
for r in (1000, 500):
    for mode in ('antialias', 'naive'):
        print(f"  {r:>4} Hz {mode:<11} {srow[f'verdict_{mode}_{r}']:>10} "
              f"{str(srow[f'flip_{mode}_{r}']):>7}")

## 7. Verdict for this trajectory

Pulling both criteria together. The rule of thumb:
- **insignificant** at rate r if the reconstruction error is small (near the interpolation floor) for every family **and** no verdict flips.
- **significant** if the error climbs sharply (commonly Mz/Fpar at 500 Hz for high-slip trajectories) or any verdict flips — and if naive >> anti-aliased, the loader must anti-alias filter.

Remember this is **one** trajectory. The fleet-level decision comes from `sampling_sensitivity.py` over the whole sweep: per-profile recon-error distributions and flip rates. The expectation to confirm there is *1000 Hz safe globally; 500 Hz acceptable only for low-slip profiles.*

In [ ]:
FLOOR = 0.05      # placeholder normalised recon-error tolerance; set from the
                  # brief's interpolation floor on real data. Below FLOOR = insignificant.
for r in (1000, 500):
    aa = max(recon[(r, 'antialias', fam)] for fam in ss.FAMILIES)
    nv = max(recon[(r, 'naive', fam)] for fam in ss.FAMILIES)
    flips = [m for m in ('antialias', 'naive') if srow[f'flip_{m}_{r}']]
    sig = 'INSIGNIFICANT' if (aa <= FLOOR and not flips) else 'SIGNIFICANT'
    note = ''
    if nv > 3 * max(aa, 1e-9):
        note = ' | naive >> anti-aliased -> loader MUST anti-alias filter'
    print(f'{r:>4} Hz: {sig:<14} (max recon: anti-aliased {aa:.4f}, naive {nv:.4f}; '
          f'flips: {flips or "none"}){note}')
print('\n(per-trajectory only — see sampling_sensitivity.py batch summary for the fleet decision)')